In [ ]:
import pandas as pd
import numpy as np
import os
import pyotp
import pyspark 
from datetime import datetime, date, time, timedelta
import datetime as dt
from datetime import datetime
import json
import time
import requests
import pyspark.pandas as ps
# importing module 

# importing sparksession from 
# pyspark.sql module 
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType, TimestampType, LongType, FloatType
from datetime import datetime
from pyspark.sql.functions import desc, asc, like, lit, abs, col, to_date, round
import pyspark.sql.functions as f
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
# import pyspark.sql.functions as sf
from pyspark.sql.functions import monotonically_increasing_id
# import matplotlib.pyplot as plt

from kiteconnect import KiteConnect
from kiteconnect import KiteTicker
from dotenv import load_dotenv
load_dotenv()  # Load credentials from .env file (not committed to version control)
creds = {
    'user_id': os.getenv('ZERODHA_USER_ID'),
    'password': os.getenv('ZERODHA_PASSWORD'),
    'totp_key': os.getenv('ZERODHA_TOTP_KEY'),
    'api_key': os.getenv('ZERODHA_API_KEY'),
    'api_secret': os.getenv('ZERODHA_API_SECRET')
}
base_url = "https://kite.zerodha.com"
login_url = "https://kite.zerodha.com/api/login"
twofa_url = "https://kite.zerodha.com/api/twofa"
instruments_url = "https://api.kite.trade/instruments"

session = requests.Session()
response = session.post(login_url,data={'user_id':creds['user_id'],'password':creds['password']})
request_id = json.loads(response.text)['data']['request_id']
twofa_pin = pyotp.TOTP(creds['totp_key']).now()
# twofa_pin =149166
response_1 = session.post(twofa_url,data={'user_id':creds['user_id'],'request_id':request_id,'twofa_value':twofa_pin,'twofa_type':'totp'})
kite = KiteConnect(api_key=creds['api_key'])
kite_url = kite.login_url()
try:
    session.get(kite_url)
except Exception as e:
    e_msg = str(e)
#print(e_msg)
request_token = e_msg.split('request_token=')[1].split(' ')[0].split('&action')[0]
print('Successful Login with Request Token:{}'.format(request_token))
data = kite.generate_session(request_token,creds['api_secret'])
access_token =  data['access_token']

kite.set_access_token(access_token)
#Get last traded prices for Nifty and BankNifty
# Initialize Kite Ticker
kws = KiteTicker(creds['api_key'], access_token)


In [ ]:

# creating sparksession and giving 
# an app name 
spark = SparkSession.builder.appName('sparkdf').getOrCreate() 

# Define the schema for the DataFrame
schema = StructType([
    StructField("instrument_token", IntegerType(), nullable=False),
    StructField("exchange_token", StringType(), nullable=False),
    StructField("tradingsymbol", StringType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("last_price", DoubleType(), nullable=False),
    StructField("expiry", DateType(), nullable=False),
    StructField("strike", DoubleType(), nullable=False),
    StructField("tick_size", DoubleType(), nullable=False),
    StructField("lot_size", IntegerType(), nullable=False),
    StructField("instrument_type", StringType(), nullable=False),
    StructField("segment", StringType(), nullable=False),
    StructField("exchange", StringType(), nullable=False)
])

# Read Parquet files into a DataFrame
Banknifty_Options_DF_From_File = spark.read.parquet("DataFiles/Instruments/Banknifty_Options", schema=schema)
Banknifty_Futures_DF_From_File = spark.read.parquet("DataFiles/Instruments/Banknifty_Futures", schema=schema)
Nifty_Options_DF_From_File = spark.read.parquet("DataFiles/Instruments/Nifty_Options", schema=schema)
Nifty_Futures_DF_From_File = spark.read.parquet("DataFiles/Instruments/Nifty_Futures", schema=schema)

# Define the schema for the expiry DataFrame
schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("current_week_expiry", DateType(), nullable=False),
    StructField("current_month_expiry", DateType(), nullable=False),
    StructField("next_week_expiry", DateType(), nullable=False),
    StructField("next_month_expiry", DateType(), nullable=False)
])
Nifty_Banknifty_Expiries_DF = spark.read.parquet("DataFiles/Instruments/Nifty_Banknifty_Expiries", schema=schema)

In [ ]:
dbutils.fs.help()

In [ ]:

# Strike_level_Name   = 'ATM'
# No_Of_Days          = 1
# CE_OR_PE            = "CE"

# df_pandas = get_Strike_OHLC_data(
#                 Options_DF          = Banknifty_Options_DF,
#                 Name                = "BANKNIFTY",
#                 Expiry              = Banknifty_current_week_expiry,
#                 Strike_level_Name   = Strike_level_Name,
#                 CE_OR_PE            = CE_OR_PE,
#                 interval            = interval,
#                 No_Of_Days          = No_Of_Days).orderBy(asc("date")).show()


In [ ]:
# Banknifty_Options_DF.show()

def get_Strike_Token(Options_DF,Name, Expiry, Strike_level_Name, CE_OR_PE):
    return Options_DF\
    .filter(Options_DF['name'] == Name)\
    .filter(Options_DF['expiry'] == Expiry)\
    .filter(Options_DF['Strike_level_Name'] == Strike_level_Name)\
    .filter(Options_DF['instrument_type'] == CE_OR_PE).select(("instrument_token")).collect()[0]["instrument_token"]


def get_Strike_OHLC_data(Options_DF,Name, Expiry, Strike_level_Name, CE_OR_PE, interval, No_Of_Days=1, No_Of_Days_Back=0):
    if CE_OR_PE == 'CE' or CE_OR_PE == 'PE':
        Options_DF =  Options_DF\
        .filter(Options_DF['name']              == Name)\
        .filter(Options_DF['expiry']            == Expiry)\
        .filter(Options_DF['Strike_level_Name'] == Strike_level_Name)\
        .filter(Options_DF['instrument_type']   == CE_OR_PE)
        
        instrument_token =  Options_DF.collect()[0]["instrument_token"]
        strike =  Options_DF.collect()[0]["strike"]
        schema = StructType([
            StructField("date",     TimestampType(),    nullable=True),
            StructField("open",     FloatType(),        nullable=True),
            StructField("high",     FloatType(),        nullable=True),
            StructField("low",      FloatType(),        nullable=True),
            StructField("close",    FloatType(),        nullable=True),
            StructField("volume",   IntegerType(),      nullable=True),
            StructField("oi",       FloatType(),      nullable=True),
            StructField("day",      DateType(),         nullable=True)
        ])
        DF =  spark.read.parquet("DataFiles/HistoricalData/"+interval+"/"+str(Expiry)+"/"+str(instrument_token), schema=schema)\
            .filter(col("volume") > 0)\
            .distinct()
        DF.createOrReplaceTempView("DF")
        
        DF = spark.sql("""
            SELECT * FROM (
                SELECT  date,
                        open,
                        high,
                        low,
                        close,
                        oi,
                        day,
                        ((dense_rank() over (order by day DESC)) - """ + str(No_Of_Days_Back) + """) as day_num
                FROM DF
                Order by date desc
            ) P WHERE day_num > 0 and day_num <= """ + str(No_Of_Days) + """
        """)
        
        return DF.orderBy(asc("date"))\
            .withColumn("instrument_token",lit(instrument_token))\
            .withColumn("name",lit(Name))\
            .withColumn("strike",lit(strike))\
            .withColumn("expiry",lit(Expiry))\
            .withColumn("ohlc4",round((col('open')+col('high')+col('low')+col('close'))/4,2)).distinct()
    else:
        CE_Options_DF =  Options_DF\
        .filter(Options_DF['name']              == Name)\
        .filter(Options_DF['expiry']            == Expiry)\
        .filter(Options_DF['Strike_level_Name'] == Strike_level_Name)\
        .filter(Options_DF['instrument_type']   == 'CE')
        CE_instrument_token =  CE_Options_DF.collect()[0]["instrument_token"]
        
        PE_Options_DF =  Options_DF\
        .filter(Options_DF['name']              == Name)\
        .filter(Options_DF['expiry']            == Expiry)\
        .filter(Options_DF['Strike_level_Name'] == Strike_level_Name)\
        .filter(Options_DF['instrument_type']   == 'PE')
        PE_instrument_token =  PE_Options_DF.collect()[0]["instrument_token"]
        
        strike =  CE_Options_DF.collect()[0]["strike"]
        schema = StructType([
            StructField("date",     TimestampType(),    nullable=True),
            StructField("open",     FloatType(),        nullable=True),
            StructField("high",     FloatType(),        nullable=True),
            StructField("low",      FloatType(),        nullable=True),
            StructField("close",    FloatType(),        nullable=True),
            StructField("volume",   FloatType(),        nullable=True),
            StructField("oi",       FloatType(),        nullable=True),
            StructField("day",      DateType(),         nullable=True)
        ])
        CE_DF= spark.read.parquet("DataFiles/HistoricalData/"+interval+"/"+str(Expiry)+"/"+str(CE_instrument_token), schema=schema)\
            .filter(col("volume") > 0)\
            .withColumn("instrument_type",lit("CE"))\
            .distinct()
        PE_DF= spark.read.parquet("DataFiles/HistoricalData/"+interval+"/"+str(Expiry)+"/"+str(PE_instrument_token), schema=schema)\
            .filter(col("volume") > 0)\
            .withColumn("instrument_type",lit("PE"))\
            .distinct()
        CE_PE_DF = CE_DF.union(PE_DF)
        CE_PE_DF.createOrReplaceTempView("CE_PE_DF")
        # Run SQL commands
        CE_PE_DF = spark.sql("""
            SELECT * FROM (
                SELECT date,
                    round(sum(Open),2) As open,
                    round(sum(high),2) As high,
                    round(sum(low),2) As low,
                    round(sum(close),2) As close,
                    round(sum(case when instrument_type = 'CE' then close else null end),2)  As CE_close,
                    round(sum(case when instrument_type = 'PE' then close else null end),2)  As PE_close,
                    round(sum(oi),2) As oi,
                    round(sum(case when instrument_type = 'CE' then oi else null end),2)  As CE_oi,
                    round(sum(case when instrument_type = 'PE' then oi else null end),2)  As PE_oi,
                    day,
                    --dense_rank() over (order by day DESC) as day_num
                    ((dense_rank() over (order by day DESC)) - """ + str(No_Of_Days_Back) + """) as day_num
                FROM CE_PE_DF
                GROUP BY day, date
                Order by date desc
            ) P WHERE day_num > 0 and day_num <= """ + str(No_Of_Days) + """
        """)
        return CE_PE_DF.orderBy(asc("date"))\
            .withColumn("instrument_token",lit(""))\
            .withColumn("name",lit(Name))\
            .withColumn("strike",lit(strike))\
            .withColumn("expiry",lit(Expiry))\
            .withColumn("ohlc4",round((col('open')+col('high')+col('low')+col('close'))/4,2)).distinct()

In [ ]:

# Strike_level_Name   = 'ATM'
# No_Of_Days          = 1
# CE_OR_PE            = "C"
# #now get data for the token
# df_pandas = get_Strike_OHLC_data(
#                     Options_DF          = Banknifty_Options_DF,
#                     Name                = "BANKNIFTY",
#                     Expiry              = Banknifty_current_week_expiry,
#                     Strike_level_Name   = Strike_level_Name,
#                     CE_OR_PE            = CE_OR_PE,
#                     interval            = interval,
#                     No_Of_Days          = No_Of_Days,
#                     No_Of_Days_Back     = 1).orderBy(asc("date"))#.select("date","open","high","low","close","oi", "strike").show()
# display(ps.DataFrame(df_pandas).head())

In [ ]:

Banknifty_current_week_expiry   = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("current_week_expiry").collect()[0][0]
Banknifty_current_month_expiry  = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("current_month_expiry").collect()[0][0]
Banknifty_next_week_expiry      = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("next_week_expiry").collect()[0][0]
Banknifty_next_month_expiry     = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("next_month_expiry").collect()[0][0]


Nifty_current_week_expiry       = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("current_week_expiry").collect()[0][0]
Nifty_current_month_expiry      = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("current_month_expiry").collect()[0][0]
Nifty_next_week_expiry          = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("next_week_expiry").collect()[0][0]
Nifty_next_month_expiry         = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("next_month_expiry").collect()[0][0]


In [ ]:

#define functions to calculate column
def calculate_Strike_level_type(col1):
    if col1 < 0:
        return 'ITM'
    elif col1 > 0:
        return 'OTM'
    else:
        return 'ATM'
    
def calculate_Strike_level_Name(col1):
    if col1 < 0:
        return 'ATM' + "" + str.replace(str(col1),".0","")
    elif col1 > 0:
        return 'ATM' + "+" + str.replace(str(col1),".0","")
    else:
        return 'ATM'
def get_Options_DF(Options_DF_From_File, ATM_Strikes, current_week_expiry, strike_range, strikes):
#Select ATM and 9 ITM and 9 OTM strikes
    Options_DF = Options_DF_From_File.filter(Options_DF_From_File['expiry'] == current_week_expiry)\
    .filter(Options_DF_From_File["strike"] <= ATM_Strikes + (strike_range*strikes))\
    .filter(Options_DF_From_File["strike"] >= ATM_Strikes - (strike_range*strikes)).orderBy(asc("strike"))

    #add customer columns
    Options_DF = Options_DF.withColumn("ATM_Strike", lit(ATM_Strikes))
    Options_DF = Options_DF.withColumn("Strike_level", ((Options_DF["strike"]-lit(ATM_Strikes))/strike_range))

    #register functions as UDF with return type
    udfcalculate_Strike_level_type = f.udf(calculate_Strike_level_type, StringType())
    udfcalculate_Strike_level_Name = f.udf(calculate_Strike_level_Name, StringType())

    #Add customer column
    Options_DF = Options_DF.withColumn("Strike_level_type", udfcalculate_Strike_level_type(Options_DF["Strike_level"]))
    Options_DF = Options_DF.withColumn("Strike_level_Name", udfcalculate_Strike_level_Name(Options_DF["Strike_level"]))

    #Select only required columns
    Options_DF = Options_DF.select( "name", "instrument_token","expiry", "Strike_level_Name", "strike", "Strike_level_type", "lot_size", "instrument_type", "segment")
    return  Options_DF


In [ ]:
def round_to_hundred(num):
    if num % 100 >= 50:
        return num + (100 - num % 100)
    else:
        return num - (num % 100)
    
def round_to_fifty(num):
    if num % 50 >= 25:
        return num + (50 - num % 50)
    else:
        return num - (num % 50)
    

def get_ATM_Strike(instrument_token, ronding_number):
    _ltp = (kite.ltp(instrument_token)[str(instrument_token)]["last_price"])
    #get ATM Strikes
    if ronding_number == 100:
        return round_to_hundred(_ltp)
    if ronding_number == 50:
        return round_to_fifty(_ltp)
    


In [ ]:
#Get all trading instruments
# instrument=kite.instruments()

#Get Instrument tokens for Nifty and BankNifty
Banknifty_instrument_token = 260105# [instrument for instrument in instrument if instrument['name'] == 'NIFTY BANK'][0]['instrument_token']
Nifty_instrument_token = 12014082#[instrument for instrument in instrument if instrument['name'] == 'NIFTY'][0]['instrument_token']
#get ATM Strikes
Banknifty_ATM_Strikes = get_ATM_Strike(Banknifty_instrument_token,100)
Nifty_ATM_Strikes     = get_ATM_Strike(Nifty_instrument_token,50)
print("Nifty_ATM_Strikes = ", Nifty_ATM_Strikes, "Banknifty_ATM_Strikes = " , Banknifty_ATM_Strikes)


In [ ]:
interval = "3minute"

In [ ]:
from pyspark.sql.functions import col, lag,when
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

def get_ROC_added(df, col_name = "close",length = 20):

    df = df.withColumn("ROC_AVG_"+col_name,lit(0))
    columns_to_drop = []
    for i in range(1,length+1):
        df = df.withColumn("ROC_"+str(i), (col(col_name) - lag(col(col_name), i).over(Window.partitionBy("strike").orderBy("rownum"))) / lag(col(col_name), i).over(Window.partitionBy("strike").orderBy("rownum")))
        df = df.withColumn("ROC_AVG_"+col_name, (col("ROC_AVG_"+col_name) + col("ROC_"+str(i))))
        columns_to_drop.append("ROC_"+str(i))
    df = df.withColumn("ROC_AVG_"+col_name, col("ROC_AVG_"+col_name)/2)
    # df = df.drop("column1", "column2", "column3")
    df = df.drop(*columns_to_drop)
    return df

def get_chart():
    if CE_OR_PE == 'CE' or CE_OR_PE == 'PE':
        df_pandas = get_Strike_OHLC_data(
                        Options_DF          = Banknifty_Options_DF,
                        Name                = "BANKNIFTY",
                        Expiry              = Banknifty_current_week_expiry,
                        Strike_level_Name   = Strike_level_Name,
                        CE_OR_PE            = CE_OR_PE,
                        interval            = interval,
                        No_Of_Days          = No_Of_Days,
                        No_Of_Days_Back     = No_Of_Days_Back).orderBy(asc("date")).select("date","open","high","low","close","oi", "strike")
    else :
        df_pandas = get_Strike_OHLC_data(
                        Options_DF          = Banknifty_Options_DF,
                        Name                = "BANKNIFTY",
                        Expiry              = Banknifty_current_week_expiry,
                        Strike_level_Name   = Strike_level_Name,
                        CE_OR_PE            = CE_OR_PE,
                        interval            = interval,
                        No_Of_Days          = No_Of_Days,
                        No_Of_Days_Back     = No_Of_Days_Back).orderBy(asc("date")).select("date","open","high","low","close","CE_close","PE_close","oi","CE_oi","PE_oi", "strike")
    # Plot the line chart
    df_pandas = df_pandas.filter(df_pandas["date"].isNotNull())
    df_pandas = df_pandas.withColumn("rownum", monotonically_increasing_id())
    df_pandas = get_ROC_added(df_pandas, col_name = "close",length = 20)
    df_pandas = get_ROC_added(df_pandas, col_name = "CE_close",length = 20)
    df_pandas = get_ROC_added(df_pandas, col_name = "PE_close",length = 20)
    

    # Define a window for calculating the cumulative sum and count
    w = Window.orderBy('date').rowsBetween(Window.unboundedPreceding, 0)

    # Create a new DataFrame with columns for cumulative sum and count
    df_pandas = df_pandas.withColumn('cumulative_sum_close', F.sum('close').over(w))
    df_pandas = df_pandas.withColumn('cumulative_min_close', F.min('close').over(w))
    df_pandas = df_pandas.withColumn('cumulative_count_close', F.count('close').over(w))
    df_pandas = df_pandas.withColumn('cumulative_sum_oi', F.sum('oi').over(w))
    df_pandas = df_pandas.withColumn('cumulative_count_oi', F.count('oi').over(w))

    # Calculate the incremental average
    df_pandas = df_pandas.withColumn('incremental_close_avg', df_pandas['cumulative_sum_close'] / df_pandas['cumulative_count_close'])
    df_pandas = df_pandas.withColumn('incremental_oi_avg', df_pandas['cumulative_sum_oi'] / df_pandas['cumulative_count_oi'])
    df_pandas = df_pandas.withColumn('incremental_min_close', df_pandas['cumulative_min_close'])
    df_pandas = df_pandas.drop("cumulative_sum_close", "colucumulative_min_closemn2", "cumulative_count_close", "cumulative_sum_oi", "cumulative_count_oi")

    df_pandas = df_pandas.withColumn("MidPoint", lit(0)) 
    print("here")
    
    df_pandas = df_pandas.toPandas()
    print("here")
    
    df_pandas['date'] = pd.to_datetime(df_pandas['date'])
    df_pandas['rownum'] = pd.to_numeric(df_pandas['rownum'])
    df_pandas['open'] = pd.to_numeric(df_pandas['open'])
    df_pandas['high'] = pd.to_numeric(df_pandas['high'])
    df_pandas['low'] = pd.to_numeric(df_pandas['low'])
    df_pandas['close'] = pd.to_numeric(df_pandas['close'])
    # df_pandas['ROC_AVG_close'] = pd.to_numeric(df_pandas['ROC_AVG'])
    
    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        df_pandas['CE_close'] = pd.to_numeric(df_pandas['CE_close'])
        df_pandas['PE_close'] = pd.to_numeric(df_pandas['PE_close'])
    df_pandas.sort_values(by='rownum', ascending=True, inplace=True)
    
    strike =  df_pandas.iloc[0]["strike"]
    # df_pandas = df_pandas.set_index('date').resample('3min').ffill()
    
    #Plot chart
    # Current_ATM_Strike = get_ATM_Strike(Banknifty_instrument_token,100)

    clear_output()
    plot_Chart(df_pandas.head(124), strike)


In [ ]:


def plot_Chart(df_pandas,strike):
    # Create a figure with two subplots

    
    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        fig = make_subplots(rows=2, cols=2)
        fig2 = make_subplots(rows=1, cols=2)
    else:
        fig = make_subplots(rows=2, cols=1)
        fig2 = make_subplots(rows=1, cols=1)
        
    # Add the 'close' line chart to the first subplot
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['close'], mode='lines', name='Staddle Close', line=dict(color='black', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['incremental_close_avg'], mode='lines', name='Staddle AVG', line=dict(color='gray', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=(df_pandas['incremental_min_close']+df_pandas['incremental_close_avg'])/2, mode='lines', name='Trailing SL', line=dict(color='red', width=2)),
        row=1, col=1
    )


    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        # Add the 'CE close' line chart to the first subplot
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['CE_close'], mode='lines', name='CALL', line=dict(color='red', width=2)),
            row=1, col=2
        )
        # Add the 'PE close' line chart to the first subplot
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['PE_close'], mode='lines', name='PUT', line=dict(color='green', width=2)),
            row=1, col=2
        )

        
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['MidPoint'], mode='lines', name='MidLine'),
            row=2, col=2
        )
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_CE_close'], mode='lines', name='CE_ROC_AVG', line=dict(color='red', width=2)),
            row=2, col=2
        )

        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_PE_close'], mode='lines', name='PE_ROC_AVG', line=dict(color='green', width=2)),
            row=2, col=2
        )
    
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['MidPoint'], mode='lines', name='MidLine'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_close'], mode='lines', name='ROC_AVG', line=dict(color='black', width=2)),
        # go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_close'], mode="lines+text", name="ROC AVG", text=['ROC_AVG_close'],textposition="top center", line=dict(color='black', width=2)),
        row=2, col=1
    )
    
    # vertical_line_x = [125]
    # fig.add_trace(go.Scatter(x=[vertical_line_x], y=[None, None], line_width=2, line_color='black', line_dash='dash'))

    # Add the 'open interest' line chart to the second subplot
    fig2.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['oi'], mode='lines', name='Combined OI'),
        row=1, col=1
    )

    fig2.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['incremental_oi_avg'], mode='lines', name='Staddle OI AVG', line=dict(color='gray', width=2)),
        row=1, col=1
    )
    # trace3 = go.Scatter(
    #     x=x,
    #     y=y3,
    #     mode='lines',
    #     name='Tapjoy',
    #     line=dict(color='purple', width=2)
    # )
    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        # Add the 'PE_OI' line chart to the second subplot
        fig2.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['CE_oi'], mode='lines', name='CALL OI'),
            row=1, col=2
        )
        # Add the 'CE_OI' line chart to the second subplot
        fig2.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['PE_oi'], mode='lines', name='PUT OI'),
            row=1, col=2
        )
    # Update the layout and show the figure
    fig.update_layout(height=600, width=1200, title_text="For Stike Level: (" +Strike_level_Name + ") / " + str(strike) + "          Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %I:%M %p")))
    fig.update_layout(paper_bgcolor='gray')
    fig.update_layout(margin=dict(l=50, r=50, b=50, t=50))
    fig.show()
    fig2.update_layout(height=400, width=1200, title_text="For Stike Level: (" +Strike_level_Name + ") / " + str(strike) + "          Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %I:%M %p")))
    fig2.update_layout(paper_bgcolor='gray')
    fig2.show()

    fig3 = go.Figure(data=[go.Candlestick(x=df_pandas['date'],
                    open=df_pandas['open'],
                    high=df_pandas['high'],
                    low=df_pandas['low'],
                    close=df_pandas['close'])])

    fig3.update_layout(height=600, width=1200, title_text="For Stike Level: (" +Strike_level_Name + ") / " + str(strike) + "          Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %H:%M")))
    fig3.update_layout(xaxis_rangeslider_visible=False)
    fig3.show()



In [ ]:
#Banknifty ATM Strike data

#For Banknifty Current Week Expiry
# Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, get_ATM_Strike(Banknifty_instrument_token,100) ,Banknifty_current_week_expiry,100,9)

# #For Nifty Current Week Expiry
# Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, get_ATM_Strike(Nifty_instrument_token,50), Nifty_current_week_expiry, 50,9)


BankNifty_Use_Custom_Strike=True
BankNifty_Custom_Strike=48900
Nifty_Use_Custom_Strike=True
Nifty_Custom_Strike=22950

#For Banknifty Current Week Expiry
if BankNifty_Use_Custom_Strike:
    Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, BankNifty_Custom_Strike ,Banknifty_current_week_expiry,100,9)
else:
    Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, get_ATM_Strike(Banknifty_instrument_token,100) ,Banknifty_current_week_expiry,100,9)
#For Nifty Current Week Expiry
if Nifty_Use_Custom_Strike:
    Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, Nifty_Custom_Strike, Nifty_current_week_expiry, 50,9)
else:
    Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, get_ATM_Strike(Nifty_instrument_token,50), Nifty_current_week_expiry, 50,9)


In [ ]:
import schedule
import time
from IPython.display import clear_output

In [ ]:


#now get data for the token
Strike_level_Name   = 'ATM'
No_Of_Days          = 1
No_Of_Days_Back     = 0
CE_OR_PE            = "E"
def job():
    # Clear the output of the current cell
    clear_output()
    # print(" Data Pulled at : " + str(datetime.now()))
    get_chart()
job()
schedule.clear()
# Schedule the job to run every 3 minutes
schedule.every(1).minutes.do(job)


# # Start the job at 9:15 am each day
# schedule.every().day.at("09:15").do(job)

# Run the scheduled jobs
while True:
    
    schedule.run_pending()
    time.sleep(5)

In [ ]:
#now get data for the token
Strike_level_Name   = 'ATM+2'
No_Of_Days          = 1
No_Of_Days_Back     = 1
CE_OR_PE            = "E"
get_chart()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[*]").appName("test").getOrCreate()
sc = spark.sparkContext
print('Default Parallelism :', sc.defaultParallelism)


In [ ]:
spark.conf.get("spark.sql.files.maxPartitionBytes")